In [ ]:
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score, classification_report

# Veriyi yükle
data = joblib.load('../data/processed/cleaned_data.joblib')
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

# Sınıf dengesizliği oranını hesapla (XGBoost scale_pos_weight için)
# Negatif Sınıf (0) Sayısı / Pozitif Sınıf (1) Sayısı
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

# Sadece ağaç modellerini yarıştırıcam (Logistic Regression'ı eledik)
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42, scale_pos_weight=ratio)
}

results = {}

for name, model in models.items():
    print(f"--- {name} Eğitiliyor... ---")
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # EN İYİ EŞİK DEĞERİNİ BULMA (Threshold Tuning)
    best_threshold = 0.5
    best_f1 = 0
    
    # 0.1'den 0.9'a kadar tüm ihtimalleri dene
    for thresh in np.arange(0.1, 0.9, 0.05):
        y_pred_temp = (y_proba >= thresh).astype(int)
        temp_f1 = f1_score(y_test, y_pred_temp)
        if temp_f1 > best_f1:
            best_f1 = temp_f1
            best_threshold = thresh
            
    # En iyi threshold ile nihai tahmin
    y_pred_best = (y_proba >= best_threshold).astype(int)
    auc = roc_auc_score(y_test, y_proba)
    
    results[name] = {
        "Optimal Threshold": round(best_threshold, 2),
        "F1 Skoru": round(best_f1, 4), 
        "ROC-AUC": round(auc, 4)
    }
    
    print(f"{name} -> Optimal Threshold: {best_threshold:.2f} | En İyi F1: {best_f1:.4f}\n")

display(pd.DataFrame(results).T)

--- Random Forest Eğitiliyor... ---
Random Forest -> Optimal Threshold: 0.10 | En İyi F1: 0.2688

--- XGBoost Eğitiliyor... ---
XGBoost -> Optimal Threshold: 0.20 | En İyi F1: 0.2425



,Optimal Threshold,F1 Skoru,ROC-AUC
Random Forest,0.1,0.2688,0.8194
XGBoost,0.2,0.2425,0.7879


In [3]:
import joblib
# Dün kaydettiğimiz model dosyasını yükleyelim
model_info = joblib.load('../models/best_fraud_model.joblib')

# Modelin tam olarak hangi sütunları beklediğini görelim
print("--- MODELİN BEKLEDİĞİ SÜTUNLAR ---")
print(model_info['features'])

--- MODELİN BEKLEDİĞİ SÜTUNLAR ---
['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed', 'Sex', 'MaritalStatus', 'Age', 'Fault', 'PolicyType', 'VehicleCategory', 'VehiclePrice', 'Deductible', 'DriverRating', 'Days_Policy_Accident', 'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'PoliceReportFiled', 'WitnessPresent', 'AgentType', 'NumberOfSuppliments', 'AddressChange_Claim', 'NumberOfCars', 'BasePolicy']


In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced']
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print("En iyi parametreler:", grid_search.best_params_)

y_proba = best_rf.predict_proba(X_test)[:, 1]

best_threshold, best_f1 = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.05):
    y_pred_temp = (y_proba >= thresh).astype(int)
    temp_f1 = f1_score(y_test, y_pred_temp)
    if temp_f1 > best_f1:
        best_f1 = temp_f1
        best_threshold = thresh

y_pred_final = (y_proba >= best_threshold).astype(int)
print(f"\nOptimize RF — F1: {best_f1:.4f} | AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"Optimal Threshold: {best_threshold:.2f}")
print("\n", classification_report(y_test, y_pred_final))

Fitting 3 folds for each of 12 candidates, totalling 36 fits
En iyi parametreler: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}

Optimize RF — F1: 0.2431 | AUC: 0.8089
Optimal Threshold: 0.50

               precision    recall  f1-score   support

           0       0.98      0.71      0.83      2899
           1       0.14      0.76      0.24       185

    accuracy                           0.72      3084
   macro avg       0.56      0.74      0.53      3084
weighted avg       0.93      0.72      0.79      3084



In [2]:
!pip install xgboost imbalanced-learn


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
